# Deploy Strands Decider 2B on SageMaker AI with network isolation

Deploy the same pinned [Strands Decider 2B checkpoint](https://huggingface.co/StrandsAgents/strands-decider-2B-hobson-v19)
and adapter as [the default notebook](deploy_strands_decider_2b_sagemaker.ipynb), with
`EnableNetworkIsolation=True`. The model container has no network access: it does not
reach PyPI, the PyTorch wheel index, GitHub, or Hugging Face at startup.

Instead, this notebook stages the Decider runtime wheel and the pinned model files once, from
the notebook environment, into Amazon S3. SageMaker AI copies them into the container
before it starts. Use this path when your endpoints cannot reach the public internet,
or when you need the deployment to use only reviewed, versioned artifacts.

It uses the same serving image as the default notebook, built on the maintained AWS
PyTorch 2.14 SageMaker DLC, so PyTorch and the Hugging Face libraries are already in the
image. Read [README.md](README.md) for prerequisites and model licenses. Run the cleanup
cell when finished.

## 1. Environment preparation

In [ ]:
%pip install --upgrade 'boto3>=1.40,<2' 'huggingface_hub>=1.5,<2' --quiet

In [ ]:
import json
import math
import os
import statistics
import time
import uuid
from pathlib import Path

import boto3
from boto3.s3.transfer import TransferConfig
from botocore.config import Config
from botocore.exceptions import ClientError

session = boto3.Session()
region = session.region_name or "us-east-1"
sm = session.client("sagemaker", region_name=region)
runtime = session.client(
    "sagemaker-runtime", region_name=region,
    config=Config(read_timeout=70, retries={"max_attempts": 0}),
)
s3 = session.client("s3", region_name=region)
iam = session.client("iam")
logs = session.client("logs", region_name=region)
identity = session.client("sts").get_caller_identity()
account_id = identity["Account"]

# Set this when the notebook's own role is not a SageMaker execution role.
execution_role = os.environ.get("SAGEMAKER_EXECUTION_ROLE_ARN")
bucket = os.environ.get("SAGEMAKER_ARTIFACT_BUCKET", f"sagemaker-{region}-{account_id}")
# Instance types in priority order. With more than one, the endpoint uses a capacity-aware
# instance pool and falls back to the next type when the first has no capacity.
instance_types = os.environ.get("SAGEMAKER_INSTANCE_TYPES", "ml.g5.xlarge,ml.g6.xlarge").split(",")
# Set this to reuse a serving image you already built; otherwise section 2 builds one.
inference_image = os.environ.get("SAGEMAKER_INFERENCE_IMAGE_URI")
codebuild_role_arn = os.environ.get("CODEBUILD_ROLE_ARN")
name = f"strands-decider-2b-iso-{uuid.uuid4().hex[:8]}"
artifact_prefix = f"strands-decider/{name}/"
example_dir = Path.cwd()
stage_dir = example_dir / f"{name}-stage"
for filename in ["inference.py", "package_offline.py", "requirements.txt", "example-request.json"]:
    if not (example_dir / filename).is_file():
        raise FileNotFoundError(f"Run the notebook from its example directory: missing {filename}")
print(f"Region: {region}; instances: {', '.join(instance_types)}; endpoint: {name}")

### Resolve the execution role

In Studio the current role may already be appropriate. Outside Studio,
set `SAGEMAKER_EXECUTION_ROLE_ARN` to an existing SageMaker role. This cell
reads the role's actual ARN through IAM, preserving paths such as
`service-role/`, and checks its SageMaker trust.

In [ ]:
if execution_role is None:
    caller_arn = identity["Arn"]
    if ":assumed-role/" not in caller_arn:
        raise ValueError("Set SAGEMAKER_EXECUTION_ROLE_ARN to an existing SageMaker execution role.")
    role_name = caller_arn.split(":assumed-role/", 1)[1].split("/", 1)[0]
else:
    role_name = execution_role.rsplit("/", 1)[-1]
role_info = iam.get_role(RoleName=role_name)["Role"]
if execution_role is not None and execution_role != role_info["Arn"]:
    raise ValueError("The configured execution-role ARN does not match the resolved IAM role.")
execution_role = role_info["Arn"]
trusts_sagemaker = any(
    statement.get("Effect") == "Allow"
    and "sagemaker.amazonaws.com" in (
        [statement.get("Principal", {}).get("Service")]
        if isinstance(statement.get("Principal", {}).get("Service"), str)
        else statement.get("Principal", {}).get("Service", [])
    )
    for statement in role_info["AssumeRolePolicyDocument"]["Statement"]
)
if not trusts_sagemaker:
    raise ValueError(
        "The current role does not trust SageMaker. Set SAGEMAKER_EXECUTION_ROLE_ARN "
        "to an appropriate existing execution role and rerun setup."
    )
print("Execution role resolved and SageMaker trust verified.")

## 2. Build the serving image

There is no AWS PyTorch inference DLC after 2.6, and that image is past its end of
support. This example runs on the maintained PyTorch 2.14 SageMaker DLC through the small
shared image in [03-features/decision-model-serving-image](../../../03-features/decision-model-serving-image/).
It adds `serve`, which answers `/ping` and `/invocations` by calling this example's
`model_fn` and `transform_fn`.

AWS CodeBuild builds the image and pushes it to a private Amazon ECR repository in your
account, in about 5 minutes. Set `CODEBUILD_ROLE_ARN` to a CodeBuild service role that can
read the artifact bucket, write build logs, and push to Amazon ECR. To reuse an image you
already built, set `SAGEMAKER_INFERENCE_IMAGE_URI` and this cell skips the build.

In [ ]:
import sys

if inference_image is None:
    if not codebuild_role_arn:
        raise ValueError("Set CODEBUILD_ROLE_ARN, or SAGEMAKER_INFERENCE_IMAGE_URI to reuse an image.")
    sys.path.insert(0, str(example_dir.parents[2] / "03-features" / "decision-model-serving-image"))
    from build_image import build_image

    inference_image = build_image(region=region, bucket=bucket, codebuild_role_arn=codebuild_role_arn)
print(f"Serving image: {inference_image}")

## 3. Stage the runtime and model files

`package_offline.py` builds one directory from this example's `inference.py` and
`requirements.txt`, using the same pins as the default deployment:

- `code/wheels/`: a wheel of the official Decider runtime built from the pinned source
  commit, and `huggingface_hub` at this notebook's version. PyTorch and the other
  libraries are already in the serving image
- `code/requirements.txt`: installs those wheels with `--no-index` from `code/wheels/`
- `code/provenance.json`: the pinned source URL and SHA-256 of the Decider wheel, which
  the adapter checks before it loads the model
- `hf-cache/`: the pinned checkpoint and base revision in Hugging Face cache layout,
  read with `HF_HUB_OFFLINE=1`

Staging downloads about 4.6 GB of weights and needs about 5 GB of free disk. It runs here, not in the
endpoint, so the notebook environment needs internet access; the endpoint does not.

In [ ]:
import package_offline

staged = package_offline.stage(example_dir, stage_dir)
print(f"Staged {staged['files']} files, {staged['bytes'] / 1e9:.1f} GB; "
      f"Decider wheel {staged['wheel']} (sha256 {staged['sha256'][:12]}…)")

### Upload the staged directory

The directory is uploaded uncompressed. SageMaker AI copies an uncompressed S3 prefix into
`/opt/ml/model/` before the container starts, which avoids packing and unpacking several
gigabytes of weights. The default artifact bucket is shared and is preserved during cleanup.

In [ ]:
try:
    s3.head_bucket(Bucket=bucket)
except ClientError as error:
    if error.response["Error"]["Code"] not in {"404", "NoSuchBucket", "NotFound"}:
        raise
    create_args = {"Bucket": bucket}
    if region != "us-east-1":
        create_args["CreateBucketConfiguration"] = {"LocationConstraint": region}
    s3.create_bucket(**create_args)
    s3.put_public_access_block(
        Bucket=bucket,
        PublicAccessBlockConfiguration={
            "BlockPublicAcls": True, "IgnorePublicAcls": True,
            "BlockPublicPolicy": True, "RestrictPublicBuckets": True,
        },
    )

transfer = TransferConfig(multipart_chunksize=64 * 1024 * 1024, max_concurrency=16)
uploaded_keys = []
for path in sorted(p for p in stage_dir.rglob("*") if p.is_file()):
    key = artifact_prefix + path.relative_to(stage_dir).as_posix()
    s3.upload_file(str(path), bucket, key, Config=transfer)
    uploaded_keys.append(key)
model_data_prefix = f"s3://{bucket}/{artifact_prefix}"
print(f"Uploaded {len(uploaded_keys)} objects to the artifact prefix.")

## 4. Create the network-isolated model, endpoint configuration, and endpoint

The differences from the default notebook are:

- `EnableNetworkIsolation=True` on the model
- `ModelDataSource` pointing at the uncompressed S3 prefix, instead of `ModelDataUrl`
- `HF_HUB_OFFLINE=1` and `HF_HUB_CACHE` pointing at the staged cache
- `DECIDER_WHEEL_PROVENANCE`, which the adapter uses to check the bundled runtime wheel

Everything else, including the image and timeouts, is unchanged.

In [ ]:
sm.create_model(
    ModelName=name,
    ExecutionRoleArn=execution_role,
    EnableNetworkIsolation=True,
    PrimaryContainer={
        "Image": inference_image,
        "ModelDataSource": {"S3DataSource": {
            "S3Uri": model_data_prefix, "S3DataType": "S3Prefix", "CompressionType": "None",
        }},
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
            "HF_HUB_OFFLINE": "1",
            "HF_HUB_CACHE": "/opt/ml/model/hf-cache",
            "HF_HUB_DISABLE_TELEMETRY": "1",
            "DECIDER_WHEEL_PROVENANCE": "/opt/ml/model/code/provenance.json",
            "TOKENIZERS_PARALLELISM": "false",
        },
    },
)
assert sm.describe_model(ModelName=name)["EnableNetworkIsolation"] is True
instance_choice = (
    {"InstanceType": instance_types[0]} if len(instance_types) == 1 else {
        "InstancePools": [{"InstanceType": t, "Priority": p} for p, t in enumerate(instance_types, 1)],
        "VariantInstanceProvisionTimeoutInSeconds": 1800,
    }
)
sm.create_endpoint_config(
    EndpointConfigName=name,
    ProductionVariants=[{
        "VariantName": "AllTraffic", "ModelName": name,
        "InitialInstanceCount": 1, "InitialVariantWeight": 1.0,
        "ModelDataDownloadTimeoutInSeconds": 900,
        "ContainerStartupHealthCheckTimeoutInSeconds": 1800,
        **instance_choice,
    }],
)
sm.create_endpoint(EndpointName=name, EndpointConfigName=name)
print(f"Network-isolated endpoint {name} is deploying.")

In [ ]:
sm.get_waiter("endpoint_in_service").wait(
    EndpointName=name, WaiterConfig={"Delay": 30, "MaxAttempts": 120}
)
endpoint = sm.describe_endpoint(EndpointName=name)
assert endpoint["EndpointStatus"] == "InService"
assert endpoint["EndpointConfigName"] == name
assert sum(v["CurrentInstanceCount"] for v in endpoint["ProductionVariants"]) == 1
print("Endpoint is InService with one instance.")

## 5. Verify the runtime and invoke typed questions

The same checks as the default notebook, plus a check that the adapter ran with the
Hugging Face Hub offline.

In [ ]:
def invoke(payload):
    response = runtime.invoke_endpoint(
        EndpointName=name, ContentType="application/json",
        Accept="application/json", Body=json.dumps(payload),
    )
    return json.loads(response["Body"].read())

diagnostics = invoke({"operation": "diagnostics"})
assert diagnostics["device"] == "cuda"
assert diagnostics["model_revision"] == "bb282d786bc251fd4e3068de3ada9ddbb38127cd"
assert diagnostics["code_revision"] == "75c9fd32e664954cdc18481434018aa507eee8fb"
assert diagnostics["source_url"] == (
    "https://github.com/strands-labs/strands-decider/archive/"
    "75c9fd32e664954cdc18481434018aa507eee8fb.tar.gz"
)
assert diagnostics["base_revision"] == "b1485b2fa6dfa1287294f269f5fb618e03d52d7c"
assert diagnostics["max_length"] == 4096 and diagnostics["strict_window"]
assert diagnostics["max_batch"] == 4
assert diagnostics["versions"]["torch"].startswith("2.14.")
assert diagnostics["versions"]["transformers"] == "5.17.0"
assert diagnostics["versions"]["peft"] == "0.21.0"
assert diagnostics["hf_hub_offline"] is True
print(json.dumps(diagnostics, indent=2))

### Tool-call readiness: Noul, Choice, and Score

The user requested weather without supplying a city, but a proposed tool call
guessed Seattle. Ask whether the arguments are grounded, what should happen
next, and how ready the call is. The harness still decides what to execute.

In [ ]:
request = json.loads((example_dir / "example-request.json").read_text())
result = invoke(request)
print(json.dumps(result, indent=2))

In [ ]:
assert result["model"] == diagnostics["model"]
answers = result["answers"]
assert set(answers) == {"arguments_grounded", "next_action", "readiness"}
assert answers["arguments_grounded"]["type"] == "noul"
assert 0 <= answers["arguments_grounded"]["noul"] <= 1
for key, kind in [("next_action", "choice"), ("readiness", "score")]:
    answer = answers[key]
    assert answer["type"] == kind
    assert 0 <= answer["confidence"] <= 1
    probabilities = list(answer["probabilities"].values())
    assert all(math.isfinite(p) and 0 <= p <= 1 for p in probabilities)
    assert abs(sum(probabilities) - 1) <= len(probabilities) * 0.00005 + 1e-6
assert answers["next_action"]["choice"] in request["questions"]["next_action"]["criteria"]
assert 0 <= answers["readiness"]["score"] <= 2
print("All three answer types verified.")

### Basic expected-answer smoke checks

These simple cases check the serving path. They are not a representative
benchmark of decision quality or calibration.

In [ ]:
language = invoke({
    "state": "This is a clear sentence written in English.",
    "questions": {"language": {
        "type": "choice", "instructions": "Which language is this sentence written in?",
        "criteria": {"English": "English", "French": "French", "German": "German"},
    }},
})
assert language["answers"]["language"]["choice"] == "English"

city_question = {"city_supplied": {
    "type": "noul", "instructions": "Did the user explicitly provide a city?",
}}
explicit_city = invoke({
    "state": "The user says: What is the current weather in Paris?",
    "questions": city_question,
})
missing_city = invoke({
    "state": "The user says: What is the current weather?",
    "questions": city_question,
})
assert explicit_city["answers"]["city_supplied"]["noul"] >= 0.5
assert missing_city["answers"]["city_supplied"]["noul"] < 0.5
print("Basic expected-answer checks: 3/3.")

### Repeatability and warm server latency

Three repeats give a smoke measurement, not a production latency estimate.
`latency_ms` excludes SDK/network time.

In [ ]:
repeats = [invoke(request) for _ in range(3)]
assert all(item["answers"] == result["answers"] for item in repeats)
print(f"Median warm server latency: {statistics.median(item['latency_ms'] for item in repeats):.2f} ms")

### Reject oversized state instead of silently truncating it

The oversized request has valid schema. SageMaker wraps the model's HTTP 422
in `ModelError`; its error body may be omitted.

In [ ]:
oversized_request = {
    "state": "weather city location " * 5000,
    "questions": city_question,
}
try:
    invoke(oversized_request)
except runtime.exceptions.ModelError as error:
    assert error.response.get("OriginalStatusCode") == 422, error.response
    print("Oversized state rejected with model HTTP 422.")
else:
    raise AssertionError("The strict context window did not reject oversized state.")

## 6. Cleanup

Delete this example's endpoint and wait before deleting its configuration and model.
Delete every object under its S3 prefix, including older versions when versioning is
enabled, its CloudWatch log group, and the local staging directory. This needs
`s3:ListBucketVersions` and `s3:DeleteObjectVersion` on the artifact bucket. The execution role
and shared bucket are preserved.

In [ ]:
def ignore_missing(operation, **arguments):
    try:
        return operation(**arguments)
    except ClientError as error:
        message = error.response["Error"].get("Message", "")
        error_code = error.response["Error"]["Code"]
        if error_code == "ResourceNotFoundException" or (
            error_code == "ValidationException"
            and ("Could not find" in message or "does not exist" in message)
        ):
            return None
        raise

endpoint_info = ignore_missing(sm.describe_endpoint, EndpointName=name)
if endpoint_info is not None:
    sm.delete_endpoint(EndpointName=name)
    sm.get_waiter("endpoint_deleted").wait(
        EndpointName=name, WaiterConfig={"Delay": 15, "MaxAttempts": 120}
    )
ignore_missing(sm.delete_endpoint_config, EndpointConfigName=name)
ignore_missing(sm.delete_model, ModelName=name)
import shutil

for page in s3.get_paginator("list_object_versions").paginate(Bucket=bucket, Prefix=artifact_prefix):
    for item in page.get("Versions", []) + page.get("DeleteMarkers", []):
        s3.delete_object(Bucket=bucket, Key=item["Key"], VersionId=item["VersionId"])
ignore_missing(logs.delete_log_group, logGroupName=f"/aws/sagemaker/Endpoints/{name}")
shutil.rmtree(stage_dir, ignore_errors=True)
assert ignore_missing(sm.describe_endpoint, EndpointName=name) is None
assert ignore_missing(sm.describe_endpoint_config, EndpointConfigName=name) is None
assert ignore_missing(sm.describe_model, ModelName=name) is None
print("Endpoint, configuration, model, S3 prefix, and endpoint logs cleaned up.")